In [3]:
import pandas as pd
from datetime import datetime
from datetime import timedelta
import requests
import urllib3
import json
from tqdm import tqdm

In [4]:
def convert_minguo_to_ad(date_str):
        parts = date_str.split('/')
        year = int(parts[0]) + 1911
        return f"{year}/{parts[1]}/{parts[2]}"

def fetch_tpex_stock_data(stock_code, date):
    url = "https://www.tpex.org.tw/www/zh-tw/afterTrading/tradingStock"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    payload = {
        "code": stock_code,
        "date": date,
        "id": "",
        "response": "json"
    }
    res = requests.post(url, headers=headers, data=payload, verify = False)
    data = json.loads(res.text)

    rows = data["tables"][0]["data"]
    columns = data["tables"][0]["fields"]

    df = pd.DataFrame(rows, columns=columns)

    # 處理證交所改變col名稱
    if "成交仟股" in df.columns:
        df.rename(columns = {"成交仟股": "Volume"}, inplace = True)
    elif "成交張數" in df.columns:
        df.rename(columns = {"成交張數": "Volume"}, inplace = True)

    df[["開盤", "最高", "最低", "收盤"]] = df[["開盤", "最高", "最低", "收盤"]].apply(pd.to_numeric, errors='coerce')
    df = df[["日 期", "開盤", "最高", "最低", "收盤", "Volume"]]
    df.rename(columns={
        "日 期": "Date",
        "開盤": "Open",
        "最高": "High",
        "最低": "Low",
        "收盤": "Close",
    }, inplace=True)
    df = df.replace(',', '', regex=True)
    df["Date"] = df["Date"].astype(str).str.replace(r'[^\d/]', '', regex = True)
    df["Date"] = df["Date"].apply(convert_minguo_to_ad)
    df["Date"] = pd.to_datetime(df["Date"], format="%Y/%m/%d")

    return df

def fetch_twse_monthly_data(stock_id, year, month, fallback_to_tpex = True):
    
    url = "https://www.twse.com.tw/exchangeReport/STOCK_DAY"
    params = {
        "response": "json",
        "date": f"{year}{month:02}01",
        "stockNo": stock_id
    }

    r = requests.get(url, params = params, verify = False)
    r.encoding = 'utf-8-sig'
    data = r.json()

    if data['stat'] != 'OK':
        if fallback_to_tpex:
            date = f"{year}/{month:02}/01"
            return fetch_tpex_stock_data(stock_id, date)
        return None

    df = pd.DataFrame(data['data'], columns = data['fields'])

    df['Date'] = df['日期'].apply(convert_minguo_to_ad)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y/%m/%d')
    df.drop(columns=['日期'], inplace=True)
    df = df.rename(columns={
        '日期': 'Date',
        '開盤價': 'Open',
        '最高價': 'High',
        '最低價': 'Low',
        '收盤價': 'Close',
    })
    df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
    df = df.replace(',', '', regex = True)
    df[['Open', 'High', 'Low', 'Close', 'Volume']] = df[['Open', 'High', 'Low', 'Close', 'Volume']].apply(pd.to_numeric, errors='coerce')

    return df


def get_tw_stock_by_twse(stock_id, start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    all_data = []
    current = start.replace(day = 1)

    while current <= end:
        df = fetch_twse_monthly_data(stock_id, current.year, current.month)
        if df is not None:
            df['Date'] = pd.to_datetime(df['Date']) 
            df = df[(df['Date'] >= start) & (df['Date'] <= end)]
            all_data.append(df)
        current += timedelta(days=32)
        current = current.replace(day=1)

    all_data = [df for df in all_data if df is not None and not df.empty]
    if not all_data:
         return None

    result = pd.concat(all_data).sort_values('Date').set_index('Date')
    return result

In [5]:
def get_prices(underlyings: list, **kwargs: dict):
    """
    get underlying's Close
    """
    prices = {}

    for underlying in tqdm(underlyings):
        data = get_tw_stock_by_twse(underlying, kwargs["start"], kwargs["end"])
        if data is None:
            print(f"{underlying} is None")
            continue
        Close = data["Close"]

        prices[underlying] = Close

    prices_df = pd.DataFrame(prices)

    return prices_df

In [6]:
hard_ware = ["3128", "5351", "5371", "6140", "6739", "3297", "3227"] # hard_ware
soft_ware = ["3260", "8299", "3339", "3529", "6643", "8054", "3324"] # soft_ware
energy = ["3122", "6130", "6138", "6291","5536", "6125", "6244"] # energy
bond = ["00679B"]

params = {
    "start": "2022-01-01",
    "end": "2025-01-01"
}

In [7]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

hard_ware_prices = get_prices(hard_ware, **params)
soft_ware_prices = get_prices(soft_ware, **params)
energy_prices = get_prices(energy, **params)
bond_prices = get_prices(bond, **params)

hard_ware_prices.to_csv("hard_ware_prices.csv")
soft_ware_prices.to_csv("soft_ware_prices.csv")
energy_prices.to_csv("energy_prices.csv")
bond_prices.to_csv("bond_prices.csv")

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:07<00:00,  7.13s/it]
